In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [9]:
files = [
    "CRMLSSold202505.csv",
    "CRMLSSold202506.csv",
    "CRMLSSold202507.csv",
    "CRMLSSold202508.csv",
    "CRMLSSold202509.csv",
    "CRMLSSold202510.csv",
    "CRMLSSold202511.csv",
    "CRMLSSold202512.csv",
    "CRMLSSold202601.csv",
    "CRMLSSold202602.csv",
    "CRMLSSold202603.csv",
    "CRMLSSold202604.csv",
    "CRMLSSold202605.csv",
    "CRMLSSold202606.csv"
]

dfs = [pd.read_csv(file) for file in files]

df = pd.concat(dfs, ignore_index=True)

/tmp/ipykernel_925/3634033777.py:18: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file) for file in files]
/tmp/ipykernel_925/3634033777.py:18: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file) for file in files]


In [10]:
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
]


### Handling missing values (before split)








In [11]:
# Drop columns where more than 70% of values are missing
threshold = 0.7
df = df.loc[:, df.isnull().mean() < threshold]

In [12]:
# drop rows that have missing values in the ClosePrice column
df = df.dropna(subset=["ClosePrice"])

# Remove observations where ClosePrice is less than or equal to 0
df = df[df['ClosePrice'] > 0]

In [13]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

In [14]:
# Remove observations with zero or negative LivingArea or LotSizeArea
df = df[(df['LivingArea'] > 0) & (df['LotSizeArea'] > 0)]


In [15]:
# Remove observations where LivingArea > 0 but BedroomsTotal or BathroomsTotalInteger are 0
# or LivingArea == 0 but BedroomsTotal or BathroomsTotalInteger are > 0

df = df[
    ((df['LivingArea'] == 0) & (df['BedroomsTotal'] == 0) & (df['BathroomsTotalInteger'] == 0)) |
    ((df['LivingArea'] > 0) & (df['BedroomsTotal'] >= 1) & (df['BathroomsTotalInteger'] >= 1))
]

In [16]:
# Assuming valid Latitude is not 0 (except for equator, which is unlikely to be a common default missing value)
# Assuming valid Longitude is between -180 and 180 (and not 0 as a default missing value)
df = df[
    (df['Latitude'] != 0) &
    (df['Longitude'] != 0) &
    (df['Longitude'] >= -180) &
    (df['Longitude'] <= 180)]

### Create train/test split

In [17]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])
df["Month"] = df["CloseDate"].dt.to_period("M")

months = sorted(df["Month"].unique())

X = 12

test_month = months[-1]
train_months = months[-(X + 1):-1]

train_df = df[df["Month"].isin(train_months)].copy()
test_df = df[df["Month"] == test_month].copy()

### Missing Value Handling on Training Data only:





In [18]:
# AttachedGarageYN column
train_df["AttachedGarageYN_missing"] = (
    train_df["AttachedGarageYN"].isna().astype(int)
)

test_df["AttachedGarageYN_missing"] = (
    test_df["AttachedGarageYN"].isna().astype(int)
)

garage_mode = train_df["AttachedGarageYN"].mode()[0]

train_df["AttachedGarageYN"] = (
    train_df["AttachedGarageYN"].fillna(garage_mode)
)

test_df["AttachedGarageYN"] = (
    test_df["AttachedGarageYN"].fillna(garage_mode)
)

/tmp/ipykernel_925/1370537505.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df["AttachedGarageYN"].fillna(garage_mode)
/tmp/ipykernel_925/1370537505.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_df["AttachedGarageYN"].fillna(garage_mode)


In [19]:
# numeric columns
# then, fill missing values with the Median
core_features = ["LivingArea", "LotSizeArea", "BedroomsTotal", "BathroomsTotalInteger"]

for col in core_features:

    train_df[col] = pd.to_numeric(
        train_df[col], errors="coerce"
    )

    test_df[col] = pd.to_numeric(
        test_df[col], errors="coerce"
    )

    median_value = train_df[col].median()

    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)


In [20]:
# replace missing values in the City and CountyOrParish columns with "Unknown"
train_df["City"] = train_df["City"].fillna("Unknown")
test_df["City"] = test_df["City"].fillna("Unknown")

train_df["CountyOrParish"] = train_df["CountyOrParish"].fillna("Unknown")
test_df["CountyOrParish"] = test_df["CountyOrParish"].fillna("Unknown")

In [21]:
# replace missing values of the YearBuilt column with median
# then, create a new column for age of the properties

train_df["YearBuilt_missing"] = (
    train_df["YearBuilt"].isna().astype(int)
)

test_df["YearBuilt_missing"] = (
    test_df["YearBuilt"].isna().astype(int)
)

year_median = train_df["YearBuilt"].median()

train_df["YearBuilt"] = train_df["YearBuilt"].fillna(year_median)
test_df["YearBuilt"] = test_df["YearBuilt"].fillna(year_median)

train_df["Age"] = 2026 - train_df["YearBuilt"]
test_df["Age"] = 2026 - test_df["YearBuilt"]

In [22]:
# remove rows with missing values in Latitude and Longitude columns
train_df = train_df.dropna(subset=["Latitude", "Longitude"]).copy()
test_df = test_df.dropna(subset=["Latitude", "Longitude"]).copy()

### Add School District Information

In [23]:
# Install geopandas library for spatial operations
!pip install geopandas

In [24]:
import geopandas as gpd
from shapely.geometry import Point

geojason_path = 'DistrictAreas2526_-284845464123469011.geojson'

# Read the GeoJSON into a GeoDataFrame from the local path
school_districts = gpd.read_file(geojason_path)

# Filter the school district dataset to only include DistrictType = "Unified"
unified_school_districts = school_districts[school_districts["DistrictType"] == "Unified"].copy()

# Ensure the CRS is set for both dataframes (WGS84 is common for lat/lon)
unified_school_districts = unified_school_districts.to_crs("EPSG:4326")

In [25]:
# Convert each property’s Latitude and Longitude into a geographic point
# for train_df
geometry_train = [Point(xy) for xy in zip(train_df["Longitude"], train_df["Latitude"])]
geo_train_df = gpd.GeoDataFrame(train_df, geometry=geometry_train, crs="EPSG:4326")

# for test_df
geometry_test = [Point(xy) for xy in zip(test_df["Longitude"], test_df["Latitude"])]
geo_test_df = gpd.GeoDataFrame(test_df, geometry=geometry_test, crs="EPSG:4326")

In [26]:
# Perform a spatial join to determine which Unified School District polygon contains each property
# For train_df
train_df_enriched = gpd.sjoin(geo_train_df, unified_school_districts, how="left", predicate="within")

# For test_df
test_df_enriched = gpd.sjoin(geo_test_df, unified_school_districts, how="left", predicate="within")

In [27]:
# Add the resulting DistrictName as a new column in dataset
# For train_df
train_df["DistrictName"] = train_df_enriched["DistrictName"].fillna("Unknown")

# For test_df
test_df["DistrictName"] = test_df_enriched["DistrictName"].fillna("Unknown")

# Clean up temporary columns from the spatial join
train_df = train_df.drop(columns=['geometry'], errors='ignore')
test_df = test_df.drop(columns=['geometry'], errors='ignore')

# Drop additional columns created by sjoin (e.g., 'index_right')
train_df = train_df.drop(columns=[col for col in train_df.columns if '_right' in col or 'index_' in col], errors='ignore')
test_df = test_df.drop(columns=[col for col in test_df.columns if '_right' in col or 'index_' in col], errors='ignore')

### Convert categorical fields to numeric (encoding)

In [28]:
# One-hot encode DistrictName
train_df = pd.get_dummies(train_df, columns=['DistrictName'], drop_first=True)
test_df = pd.get_dummies(test_df, columns=['DistrictName'], drop_first=True)

# Align columns after one-hot encoding, as some districts might only appear in train or test
# Reindex test_df to match train_df columns, filling new columns with 0
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

In [29]:
# city frequency encoding
city_counts = train_df["City"].value_counts()

common_cities = city_counts[
    city_counts >= 50
].index


train_df["City"] = train_df["City"].where(
    train_df["City"].isin(common_cities),
    "Other"
)

test_df["City"] = test_df["City"].where(
    test_df["City"].isin(common_cities),
    "Other"
)

In [30]:
# One-hot encoding
train_df = pd.get_dummies(
    train_df,
    columns=["City", "CountyOrParish"],
    drop_first=True
)

test_df = pd.get_dummies(
    test_df,
    columns=["City", "CountyOrParish"],
    drop_first=True
)

In [31]:
# Make test columns exactly match train columns
# (important because test may miss some cities)
test_df = test_df.reindex(
    columns=train_df.columns,
    fill_value=0
)

In [32]:
X_train = train_df.drop(columns=["ClosePrice"])
y_train = train_df["ClosePrice"]

X_test = test_df.drop(columns=["ClosePrice"])
y_test = test_df["ClosePrice"]

### cleaned sets:

In [33]:
train_df.to_csv("train_preprocessed.csv", index=False)
test_df.to_csv("test_preprocessed.csv", index=False)

In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error
)
from sklearn.preprocessing import StandardScaler

## Load Test and Training Sets

In [35]:
train_df = pd.read_csv("train_preprocessed.csv")

test_df = pd.read_csv("test_preprocessed.csv")

/tmp/ipykernel_925/2367302725.py:1: DtypeWarning: Columns (43,51) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("train_preprocessed.csv")
/tmp/ipykernel_925/2367302725.py:3: DtypeWarning: Columns (43,51) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv("test_preprocessed.csv")


In [36]:
city_cols = [col for col in train_df.columns if col.startswith("City_")]
county_cols = [col for col in train_df.columns if col.startswith("CountyOrParish_")]
district_cols = [col for col in train_df.columns if col.startswith("DistrictName_")]

features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeArea",
    "Age",
    "Latitude",
    "Longitude"
] + city_cols + county_cols + district_cols

In [37]:
# Separate predictors and target variable for training and testing sets

X_train = train_df[features]
y_train = train_df['ClosePrice']
y_train_log = np.log1p(y_train)

X_test = test_df[features]
y_test = test_df['ClosePrice']


## Function to calculate metrics:

In [38]:
def evaluate_model(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)

    mae = mean_absolute_error(y_true, y_pred)

    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    ape = np.abs((y_true - y_pred) / y_true) * 100
    mdape = np.median(ape)

    return r2, mae, mape, mdape

## Fit Models

In [39]:
# Baseline: Linear Regression

model = LinearRegression()
model.fit(X_train, y_train)
lr_pred = model.predict(X_test)


# Decision Tree Regressor
dt_model = DecisionTreeRegressor(
    max_depth=20,
    min_samples_leaf=5,
    random_state=42
)

dt_model.fit(X_train, y_train_log)

dt_pred_log = dt_model.predict(X_test)

dt_pred = np.expm1(dt_pred_log)


In [40]:
# Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train_log)

rf_pred_log = rf_model.predict(X_test)

# Convert back to dollars
rf_pred = np.expm1(rf_pred_log)

In [41]:
# Linear Regression
lr_metrics = evaluate_model(y_test, lr_pred)

# Decision Tree
dt_metrics = evaluate_model(y_test, dt_pred)

# Random Forest
rf_metrics = evaluate_model(y_test, rf_pred)

## Put results into a table

In [42]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "R²": [
        lr_metrics[0],
        dt_metrics[0],
        rf_metrics[0]
    ],
    "MAE": [
        lr_metrics[1],
        dt_metrics[1],
        rf_metrics[1]
    ],
    "MAPE (%)": [
        lr_metrics[2],
        dt_metrics[2],
        rf_metrics[2]
    ],
    "MdAPE (%)": [
        lr_metrics[3],
        dt_metrics[3],
        rf_metrics[3]
    ]
})

results

,Model,R²,MAE,MAPE (%),MdAPE (%)
0,Linear Regression,-103.585483,625803.710579,87.547707,27.499505
1,Decision Tree,0.748610,254873.068912,26.224400,10.953916
2,Random Forest,0.766011,218901.785088,22.273164,9.131458
